# ships along the Seattle shoreline

animation for identical, unmoored ships placed along the full 4.4 km waterfront

ships collide with buildings but do not collide with each other

## Setup

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
from netCDF4 import Dataset
import numpy as np
import pandas as pd
from shapely.geometry import box, shape
from shapely.ops import transform, unary_union

HERE = Path.cwd().resolve()
if not (HERE / 'setrun.py').exists(): HERE = HERE / 'examples' / 'seattle_debris'
if not (HERE / 'setrun.py').exists(): raise FileNotFoundError('Run this notebook from the Seattle example or repository root')
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parents[1] / 'src'))
from geoclaw_debris.ship_tracking import Flow, Ship, check_pose, hull_polygon, simulate

TSUNAMI_DIR = HERE / '_output' / 'seattle_shoreline'
TSUNAMI_FILE = TSUNAMI_DIR / 'seattle_fault_fgout.nc'
TERRAIN_FILE = TSUNAMI_DIR / 'seattle_buildings_mhw.tt3'
INVENTORY_FILE = HERE.parents[1] / 'SeattleBuildings' / 'Seattle_BuildingInventory.geojson'

SHORE_BOUNDS = [-122.375, -122.325, 47.580, 47.620]
GRID_METERS = 24. # a bit coarser than usual for sake of speed
TSUNAMI_END = 1200.
FLOW_INTERVAL = 5.
SHORE_DISTANCE = (24., 60.)
SEED = 23
SHIP = Ship(length_m=30., beam_m=8., draft_m=1.5)
HEADING = 0.
SHIP_DT = 1.
TERRAIN_BOUNDS = [-122.390, -122.310, 47.560, 47.640]
REFINED_BOUNDS = [-122.385, -122.315, 47.565, 47.635]


In [ ]:
from rasterio.features import rasterize
from rasterio.transform import from_origin
from clawpack.geoclaw import fgout_tools
from prepare_topo import read_mhw_region, write_topo, file_hash

def cached_flow(path, **attributes):
    if not path.exists(): return False
    with Dataset(path) as ds: return all(getattr(ds, key, None) == value for key, value in attributes.items())

def run_solver(folder):
    print('running GeoClaw; log:', folder / 'geoclaw.log', flush=True)
    with (folder / 'geoclaw.log').open('w') as log: subprocess.run([str(HERE / 'xgeoclaw')], cwd=folder, stdout=log, stderr=subprocess.STDOUT, check=True)
    log_text = (folder / 'geoclaw.log').read_text()
    if 'Stopping calculation' in log_text: raise RuntimeError(f'GeoClaw stopped early; inspect {folder / "geoclaw.log"}')

def save_flow(path, time_units='seconds', **attributes):
    grid = fgout_tools.FGoutGrid(1, str(path.parent), 'binary32')
    grid.read_fgout_grids_data()
    for i in range(1, grid.nout+1):
        if not (path.parent / f'fgout0001.t{i:04d}').exists(): raise RuntimeError(f'Missing flow frame {i}/{grid.nout}; refusing to cache an incomplete run')
    first = grid.read_frame(1)
    temporary = path.with_suffix('.partial.nc')
    with Dataset(temporary, 'w') as ds:
        ds.setncatts(dict(terrain_sha256=terrain_hash, inventory_sha256=inventory_hash, building_height_source='NSI bldheight (m); missing heights use 20 m', **attributes))
        for name, values in [('lon', first.x), ('lat', first.y)]:
            ds.createDimension(name, len(values))
            ds.createVariable(name, 'f8', (name,))[:] = values
        ds.createDimension('time', grid.nout)
        ds.createVariable('time', 'f8', ('time',)).units = time_units
        for name in ('h', 'u', 'v', 'B'):
            var = ds.createVariable(name, 'f4', ('time', 'lon', 'lat'), zlib=True, complevel=1, chunksizes=(1, len(first.x), len(first.y)))
            var.units = 'm/s' if name in ('u', 'v') else 'm'
        for i in range(grid.nout):
            frame = first if i == 0 else grid.read_frame(i+1)
            ds['time'][i] = frame.t
            for name in ('h', 'u', 'v', 'B'): ds[name][i] = getattr(frame, name)
    temporary.replace(path)

features = json.loads(INVENTORY_FILE.read_text())['features']
inventory = pd.DataFrame([dict(geometry=shape(f['geometry']), nsi_height_m=f['properties'].get('bldheight')) for f in features])
inventory['nsi_height_m'] = pd.to_numeric(inventory.nsi_height_m, errors='coerce')
inventory['height_m'] = inventory.nsi_height_m.where(inventory.nsi_height_m > 0, 20.)
TERRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
x, y, bare = read_mhw_region(HERE / 'puget_sound_mhw_19_n47x75_w122x50_2023v1.nc', TERRAIN_BOUNDS)
region = box(x[0], y[0], x[-1], y[-1])
local = inventory[inventory.geometry.map(lambda p: p.intersects(region))].sort_values('height_m')
transform_grid = from_origin(x[0]-(x[1]-x[0])/2, y[-1]+(y[1]-y[0])/2, x[1]-x[0], y[1]-y[0])
heights = rasterize(list(zip(local.geometry, local.height_m)), out_shape=bare.shape, transform=transform_grid, fill=0, dtype='float64')[::-1]
terrain = np.where(heights > 0, np.maximum(bare, 0.)+heights, bare)
write_topo(TERRAIN_FILE, x, y, terrain)
terrain_hash = file_hash(TERRAIN_FILE)
inventory_hash = file_hash(INVENTORY_FILE)
print(f'{(local.nsi_height_m > 0).sum()}/{len(local)} footprints use NSI heights while others use default 20 m')


In [ ]:
from urllib.request import urlretrieve
from setrun import setrun

X1, X2, Y1, Y2 = SHORE_BOUNDS
TSUNAMI_DIR.mkdir(parents=True, exist_ok=True)
source = TSUNAMI_DIR / 'seattlefault_uniform.tt3'
if not source.exists():
    temporary = source.with_suffix('.partial')
    urlretrieve('https://depts.washington.edu/ptha/dtopo/seattlefault_uniform.tt3', temporary)
    temporary.replace(source)
with Dataset(HERE / 'puget_sound_mhw_19_n47x75_w122x50_2023v1.nc') as ds:
    rx, ry = np.asarray(ds['lon'][::12]), np.asarray(ds['lat'][::12])
    regional = np.asarray(ds['z'][::12, ::12])
assert np.isfinite(regional).all()
write_topo(TSUNAMI_DIR / 'regional_mhw.tt3', rx, ry, regional)
run = setrun()
c = run.clawdata
c.cfl_desired, c.cfl_max = .6, .8
dx = GRID_METERS / (111000.*np.cos(np.deg2rad((Y1+Y2)/2))) * 8
dy = GRID_METERS / 111000. * 8
c.lower = [float(X1-np.floor((X1-rx[0])/dx)*dx), float(Y1-np.floor((Y1-ry[0])/dy)*dy)]
c.num_cells = [int(np.floor((rx[-1]-c.lower[0])/dx)), int(np.floor((ry[-1]-c.lower[1])/dy))]
c.upper = [float(c.lower[0]+c.num_cells[0]*dx), float(c.lower[1]+c.num_cells[1]*dy)]
c.tfinal, c.output_style, c.output_times, c.output_t0, c.checkpt_style = TSUNAMI_END, 2, [TSUNAMI_END], False, 0
run.qinit_data.qinit_type, run.qinit_data.qinitfiles = 0, []
run.topo_data.topofiles = [[3, str(TSUNAMI_DIR / 'regional_mhw.tt3')], [3, str(TERRAIN_FILE)]]
run.dtopo_data.dtopofiles = [[3, str(source)]]
run.dtopo_data.dt_max_dtopo = .2
run.amrdata.amr_levels_max = 3
run.amrdata.refinement_ratios_x = run.amrdata.refinement_ratios_y = run.amrdata.refinement_ratios_t = [2, 4]
run.refinement_data.wave_tolerance = 1.e99
run.regiondata.regions = [[2, 2, 0., TSUNAMI_END, *TERRAIN_BOUNDS], [3, 3, 0., TSUNAMI_END, *REFINED_BOUNDS]]
fg = run.fgout_data.fgout_grids[0]
fg.x1, fg.x2, fg.y1, fg.y2 = SHORE_BOUNDS
fg.nx = round((X2-X1)*111000.*np.cos(np.deg2rad((Y1+Y2)/2))/GRID_METERS)
fg.ny = round((Y2-Y1)*111000./GRID_METERS)
fg.tend, fg.nout = TSUNAMI_END, int(TSUNAMI_END/FLOW_INTERVAL)+1
print(f'waterfront fgout: {fg.nx} x {fg.ny}, {fg.nout} frames', flush=True)
signature = hashlib.sha256((terrain_hash + inventory_hash + file_hash(source) + file_hash(TSUNAMI_DIR / 'regional_mhw.tt3') + repr((c.lower, c.upper, c.num_cells, REFINED_BOUNDS, TSUNAMI_END, GRID_METERS, FLOW_INTERVAL, fg.nx, fg.ny, "shoreline-v3-padded-2x4-cfl06"))).encode()).hexdigest()
if not cached_flow(TSUNAMI_FILE, scenario_sha256=signature):
    run.write(str(TSUNAMI_DIR))
    run_solver(TSUNAMI_DIR)
    save_flow(TSUNAMI_FILE, time_units='seconds since earthquake', scenario_sha256=signature,
              forcing='UW SF-L Seattle Fault seafloor deformation',
              source_url='https://depts.washington.edu/ptha/dtopo/seattlefault_uniform.tt3', vertical_datum='MHW')


In [ ]:
from scipy.ndimage import distance_transform_edt
from shapely.prepared import prep

with Dataset(TSUNAMI_FILE) as ds:
    flow = Flow(ds)
    domain = box(flow.lon.min(), flow.lat.min(), flow.lon.max(), flow.lat.max())
    footprints = [p for p in inventory.geometry if p.intersects(domain)]
    buildings = unary_union(footprints)
    metric_footprints = [transform(flow.to_xy, p) for p in footprints]
    obstacles = prep(transform(flow.to_xy, buildings))
    gx, gy = flow.to_xy(flow.lon, flow.lat)
    initial_depth = np.asarray(ds['h'][0])
    distance = distance_transform_edt(initial_depth > .1, sampling=(gx[1]-gx[0], gy[1]-gy[0]))
    candidates = np.argwhere((initial_depth > 3.) & (distance >= SHORE_DISTANCE[0]) & (distance <= SHORE_DISTANCE[1]))
    rng = np.random.default_rng(SEED)
    starts, hulls = [], []
    for i, j in rng.permutation(candidates):
        center = (gx[i], gy[j])
        hull = hull_polygon(center, np.deg2rad(HEADING), SHIP)
        clear, _, _ = check_pose(flow, obstacles, SHIP, center, np.deg2rad(HEADING), 0.)
        if clear and all(not hull.buffer(10).intersects(other) for other in hulls):
            starts.append(flow.to_lonlat(*center))
            hulls.append(hull)
    FLEET_SIZE = len(starts)
    print(f'placed {FLEET_SIZE} ships along the shoreline', flush=True)

    paths = []
    for k, start in enumerate(starts):
        paths.append(simulate(flow, buildings, SHIP, start, heading_deg=HEADING, end_time=TSUNAMI_END, dt=SHIP_DT))
        if (k+1) % 50 == 0: print(f'tracked {k+1}/{FLEET_SIZE} ships', flush=True)

pd.concat([pd.DataFrame(path).assign(ship=k) for k, path in enumerate(paths)]).to_csv(TSUNAMI_DIR / 'fleet_paths.csv', index=False)


In [ ]:
# white dots are tracers to visualize flow, and arrows show water movement when its over land
from collections import Counter
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from scipy.interpolate import RegularGridInterpolator
from IPython.display import Video, display

with Dataset(TSUNAMI_FILE) as ds:
    water_times = np.asarray(ds['time'][:])
    depth = np.asarray(ds['h'][:])
    u, v = np.asarray(ds['u'][:]), np.asarray(ds['v'][:])

open_water = (initial_depth > 10.) & (distance > 120.)
open_cells = np.argwhere(open_water)
open_xy = np.column_stack((gx[open_cells[:, 0]], gy[open_cells[:, 1]]))
tracer_rng = np.random.default_rng(7)
def release_tracers(count):
    return open_xy[tracer_rng.integers(len(open_xy), size=count)]

velocity = RegularGridInterpolator((water_times, gx, gy), np.stack((u, v), axis=-1), bounds_error=False, fill_value=np.nan)
open_area = RegularGridInterpolator((gx, gy), open_water.astype(float), bounds_error=False, fill_value=0.)
def water_velocity(points, time):
    return velocity(np.column_stack((np.full(len(points), time), points)))

tracer_times = np.arange(water_times[0], water_times[-1]+1, 1.)
tracer_xy = np.empty((len(tracer_times), 220, 2))
tracer_reset = np.zeros(tracer_xy.shape[:2], dtype=bool)
tracer_xy[0] = release_tracers(220)
for k, time in enumerate(tracer_times[:-1]):
    points = tracer_xy[k]
    midpoint = points + .5*water_velocity(points, time)
    proposed = points + water_velocity(midpoint, time+.5)
    valid = (open_area(midpoint) > .99) & (open_area(proposed) > .99)
    proposed[~valid] = release_tracers(np.count_nonzero(~valid))
    tracer_xy[k+1] = proposed
    tracer_reset[k+1] = ~valid

ship_times = np.array([r['time_s'] for r in paths[0]])
xy = [np.array([[r['x_m'], r['y_m']] for r in path]) for path in paths]
colors = plt.get_cmap('turbo')(np.linspace(.04, .96, FLEET_SIZE))
fig, ax = plt.subplots(figsize=(9, 9), constrained_layout=True)
extent_x, extent_y = flow.to_xy(np.array(SHORE_BOUNDS[:2]), np.array(SHORE_BOUNDS[2:]))
ax.contourf(gx, gy, initial_depth.T, levels=[-.01, .1, float(initial_depth.max())+1], colors=['#e5e0d6', '#dceaf0'])
for footprint in metric_footprints:
    for piece in getattr(footprint, 'geoms', [footprint]):
        if piece.geom_type == 'Polygon':
            ax.add_patch(PolygonPatch(np.asarray(piece.exterior.coords), facecolor='#827e76', edgecolor='none', alpha=.7, zorder=3))
ax.set(xlim=extent_x, ylim=extent_y, xlabel='east (m)', ylabel='north (m)', aspect='equal')
water = ax.pcolormesh(gx, gy, np.ma.masked_less(depth[0].T, .1), cmap='Blues', vmin=0, vmax=12, shading='auto', alpha=.7)
fig.colorbar(water, ax=ax, label='water depth (m)', shrink=.65)
arrows = ax.quiver(gx[::12], gy[::12], u[0, ::12, ::12].T, v[0, ::12, ::12].T, angles='xy', scale_units='xy', scale=.04, color='#36576b')
ax.quiverkey(arrows, .08, .96, 2., '2 m/s', labelpos='E')
tracer_trails = LineCollection([], colors='white', linewidths=.8, alpha=.65, zorder=2)
ax.add_collection(tracer_trails)
tracer_dots = ax.scatter(*tracer_xy[0].T, s=5, color='white', alpha=.85, zorder=2)
tracer_label = ax.text(.02, .02, 'white tracers: water motion', transform=ax.transAxes, color='white', fontsize=8)
hulls, trails = [], []
for color in colors:
    hull = PolygonPatch(np.zeros((4, 2)), facecolor=color, edgecolor='#222', zorder=5)
    ax.add_patch(hull)
    hulls.append(hull)
    trails.append(ax.plot([], [], color=color, lw=1, zorder=4)[0])

def update(frame):
    time = water_times[frame]
    step = np.searchsorted(ship_times, time)
    water.set_array(np.ma.masked_less(depth[frame].T, .1).ravel())
    arrows.set_UVC(np.ma.masked_where(depth[frame, ::12, ::12].T < .1, u[frame, ::12, ::12].T), np.ma.masked_where(depth[frame, ::12, ::12].T < .1, v[frame, ::12, ::12].T))
    tracer_step = np.searchsorted(tracer_times, time)
    first = max(0, tracer_step-30)
    continuous = ~tracer_reset[first+1:tracer_step+1].any(axis=0)
    tracer_dots.set_offsets(tracer_xy[tracer_step])
    tracer_trails.set_segments(tracer_xy[first:tracer_step+1, continuous].transpose(1, 0, 2))
    for k, path in enumerate(paths):
        row = path[step]
        hull = hull_polygon(xy[k][step], np.deg2rad(row['heading_deg']), SHIP)
        hulls[k].set_xy(np.asarray(hull.exterior.coords))
        trails[k].set_data(xy[k][:step+1, 0], xy[k][:step+1, 1])
    states = Counter(path[step]['status'] for path in paths)
    ax.set_title(f'seattle shoreline fleet: {time:.0f} s\n' + ', '.join(f'{s}: {n}' for s, n in states.items()))

frames = np.unique(np.r_[np.arange(0, len(water_times), 4), len(water_times)-1])
animation = FuncAnimation(fig, update, frames=frames, interval=125)
plt.close(fig)


In [ ]:
movie = TSUNAMI_DIR / 'shoreline_fleet.mp4'
animation.save(str(movie), writer='ffmpeg', fps=8, dpi=100)
Video(str(movie), embed=True)


In [ ]:
# ending positions
update(len(water_times)-1)
tracer_dots.set_visible(False)
tracer_label.set_visible(False)
tracer_trails.set_visible(False)
for trail in trails: trail.set_visible(False)
for hull in hulls: hull.set(facecolor='#ff9a28', edgecolor='white', linewidth=.3)
fig.savefig(TSUNAMI_DIR / 'shoreline_end_positions.png', dpi=180)
display(fig)
